<a href="https://colab.research.google.com/github/massimilianogasparini-author/creative-loop-dynamics/blob/main/mad_bifurcation_phase_portrait_creative_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Importazione rigorosa delle librerie matematiche e di interfaccia
import numpy as np
from scipy.integrate import odeint
from scipy.optimize import fsolve
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Funzione analitica per l'estrazione della soglia critica
# Questa funzione isola il calcolo di phi_crit per consentire ricalcoli dinamici
# nel caso in cui, in futuro, si desideri rendere interattivi anche altri parametri.
def calcola_soglia_critica(alpha, beta, gamma, lambd_, K, Ks):
    # Derivata della funzione di equilibrio phi(h)
    def dphi_dh(h):
        term1 = beta * Ks * np.exp(-(lambd_/gamma)*h) * (1 - (lambd_/gamma)*h)
        term2 = -alpha * (1 - 2*h/K)
        return term1 + term2

    # Ricerca del punto critico h_tau sul margine di sopravvivenza
    h_tau = fsolve(dphi_dh, 1.5)[0]

    # Valutazione della soglia di transizione di fase
    phi_crit = beta * h_tau * Ks * np.exp(-(lambd_/gamma)*h_tau) - alpha * h_tau * (1 - h_tau/K)
    return h_tau, phi_crit

# Motore di simulazione collegato all'interfaccia interattiva
def simulatore_interattivo_mad(phi):
    # 1. Definizione dei parametri termodinamici del sistema
    alpha, beta, gamma, lambd_, K, Ks = 0.5, 0.3, 0.4, 0.2, 10.0, 10.0

    # 2. Calcolo analitico vincolato (per ottenere il riferimento teorico)
    h_tau, phi_crit = calcola_soglia_critica(alpha, beta, gamma, lambd_, K, Ks)

    # 3. Definizione del sistema di equazioni differenziali
    def sistema_hs(y, t):
        h, s = y
        s_safe = max(s, 1e-6) # Prevenzione singolarità logaritmica

        dh_dt = alpha * h * (1 - h / K) - beta * h * s + phi
        ds_dt = gamma * s * np.log(Ks / s_safe) - lambd_ * s * h
        return [dh_dt, ds_dt]

    # 4. Integrazione Numerica
    t = np.linspace(0, 150, 2000)
    y0 = [1.0, 1.0] # Iniezione iniziale
    soluzione = odeint(sistema_hs, y0, t)

    h_dati = soluzione[:, 0]
    s_dati = soluzione[:, 1]

    # Valutazione dello stato finale (Equilibrio attrattivo)
    h_finale = h_dati[-1]
    stato_sistema = "COLLASSO INFORMAZIONALE (MAD)" if h_finale < 0.1 else "CREATIVE LOOP STABILE"
    colore_stato = "red" if h_finale < 0.1 else "green"

    # 5. Generazione del Layout Visivo
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # --- Grafico 1: Evoluzione Temporale ---
    ax1.plot(t, h_dati, 'b-', label='Informazione Umana (h)', linewidth=2.5)
    ax1.plot(t, s_dati, 'r--', label='Informazione Sintetica (s)', linewidth=2.5)
    ax1.axhline(0, color='black', linewidth=1)

    ax1.set_title('Evoluzione Temporale delle Dinamiche Informazionali', fontsize=12)
    ax1.set_xlabel('Tempo (Epoche)', fontsize=11)
    ax1.set_ylabel('Volume Informazionale', fontsize=11)
    ax1.grid(True, linestyle=':', alpha=0.7)
    ax1.legend(loc='upper right')

    # Annotazione testuale rigorosa dello stato
    ax1.text(0.05, 0.95, f'Stato: {stato_sistema}\n$\phi$ impostato: {phi:.2f}\n$\phi_{{crit}}$ teorico: {phi_crit:.3f}',
             transform=ax1.transAxes, fontsize=11, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='white', edgecolor=colore_stato, alpha=0.8))

    # --- Grafico 2: Ritratto di Fase e Geometria degli Attrattori ---
    ax2.plot(h_dati, s_dati, 'k-', label='Traiettoria del Sistema', linewidth=2)
    ax2.plot(h_dati[0], s_dati[0], 'go', markersize=8, label='Stato Iniziale')
    ax2.plot(h_dati[-1], s_dati[-1], 'ro', markersize=8, label='Attrattore Finale')

    # Tracciamento Nullcline per dimostrazione matematica
    h_vals_null = np.linspace(0.1, 15, 200)
    s_null_h = (alpha * (1 - h_vals_null/K) + phi) / beta # Nullclina dh/dt = 0

    s_vals_null = np.linspace(0.1, Ks, 200)
    h_null_s = (gamma / lambd_) * np.log(Ks / s_vals_null) # Nullclina ds/dt = 0

    ax2.plot(h_vals_null, s_null_h, 'b:', label='Nullclina dh/dt = 0', alpha=0.5)
    ax2.plot(h_null_s, s_vals_null, 'r:', label='Nullclina ds/dt = 0', alpha=0.5)

    ax2.set_title('Spazio delle Fasi (Ritratto di Fase)', fontsize=12)
    ax2.set_xlabel('Informazione Umana (h)', fontsize=11)
    ax2.set_ylabel('Informazione Sintetica (s)', fontsize=11)
    ax2.set_xlim(-0.5, 12)
    ax2.set_ylim(-0.5, 12)
    ax2.grid(True, linestyle=':', alpha=0.7)
    ax2.legend(loc='upper right')

    plt.tight_layout()
    plt.show()

# Istanziazione dell'interfaccia Ipywidgets
# Lo slider trasmette il valore flottante direttamente al risolutore
interfaccia_phi = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=5.0,
    step=0.05,
    description='Iniezione ($\phi$):',
    continuous_update=False, # Calcola al rilascio del mouse per non sovraccaricare il kernel
    layout=widgets.Layout(width='600px'),
    style={'description_width': 'initial'}
)

# Binding finale: unisce l'elemento UI alla funzione di integrazione
out = widgets.interactive_output(simulatore_interattivo_mad, {'phi': interfaccia_phi})

display(widgets.VBox([interfaccia_phi, out]))